# Day 4: Fund Performance Analytics

Comprehensive performance analysis for 40 mutual funds including:
- Daily returns computation
- CAGR for 1yr, 3yr, 5yr periods
- Sharpe and Sortino ratios
- Alpha and Beta vs Nifty 100
- Maximum Drawdown analysis
- Fund Scorecard ranking
- Benchmark comparison visualization


In [ ]:
# Section 1: Import Libraries and Configure File Paths
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
from datetime import datetime, timedelta
import warnings
import os

warnings.filterwarnings('ignore')

# Configure matplotlib for headless execution
plt.switch_backend('Agg')

# Configure plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Define file paths - handle both notebook and script execution
current_dir = Path.cwd()
print(f"Current working directory: {current_dir}")

# If we're in the project root, use it directly
if (current_dir / 'data' / 'processed').exists():
    project_root = current_dir
# If we're in the notebooks directory, go up one level
elif (current_dir.parent / 'data' / 'processed').exists():
    project_root = current_dir.parent
else:
    # Fallback to looking for the data directory
    project_root = current_dir

data_processed_dir = project_root / 'data' / 'processed'
data_raw_dir = project_root / 'data' / 'raw'
reports_dir = project_root / 'reports'
charts_dir = reports_dir / 'charts'

# Create charts directory if it doesn't exist
charts_dir.mkdir(parents=True, exist_ok=True)

print("File paths configured:")
print(f"  Project root: {project_root}")
print(f"  Data processed: {data_processed_dir}")
print(f"  Reports/Charts: {charts_dir}")
print(f"  Data dir exists: {data_processed_dir.exists()}")


In [ ]:
# Section 2: Load Processed NAV and Benchmark Datasets
nav_history = pd.read_csv(data_processed_dir / '02_nav_history_cleaned.csv')
fund_master = pd.read_csv(data_processed_dir / '01_fund_master_cleaned.csv')
benchmark_indices = pd.read_csv(data_processed_dir / '10_benchmark_indices_cleaned.csv')

# Convert date columns to datetime
nav_history['date'] = pd.to_datetime(nav_history['date'])
benchmark_indices['date'] = pd.to_datetime(benchmark_indices['date'])
fund_master['launch_date'] = pd.to_datetime(fund_master['launch_date'], errors='coerce')

# Sort data by date
nav_history = nav_history.sort_values('date').reset_index(drop=True)
benchmark_indices = benchmark_indices.sort_values('date').reset_index(drop=True)

print(f"NAV History shape: {nav_history.shape}")
print(f"Fund Master shape: {fund_master.shape}")
print(f"Benchmark Indices shape: {benchmark_indices.shape}")
print(f"\nNAV Date range: {nav_history['date'].min()} to {nav_history['date'].max()}")
print(f"Unique funds: {nav_history['amfi_code'].nunique()}")
print(f"Benchmark indices: {benchmark_indices['index_name'].unique().tolist()}")


In [ ]:
# Section 3: Compute Daily Returns for All Funds
"""
Daily return = (NAV_t / NAV_t-1) - 1
Computed for each fund independently.
"""

# Pivot NAV history to get each fund as a column
nav_pivot = nav_history.pivot_table(index='date', columns='amfi_code', values='nav')
print(f"NAV pivot shape: {nav_pivot.shape}")

# Calculate daily returns
daily_returns = nav_pivot.pct_change()

# Save returns with date as index
returns_export = daily_returns.reset_index()
returns_export.to_csv(data_processed_dir / 'returns_computed.csv', index=False)

print(f"\n✓ Daily returns computed and saved: {data_processed_dir / 'returns_computed.csv'}")
print(f"Returns matrix shape: {daily_returns.shape}")
print(f"\nDaily returns statistics (%):")

# Print statistics for first 5 funds
stats_df = (daily_returns * 100).describe().T
print(stats_df.head(10))

print(f"\nOverall statistics across all funds:")
print(f"  Mean daily return: {(daily_returns * 100).values.flatten().mean():.4f}%")
print(f"  Std dev daily return: {(daily_returns * 100).values.flatten().std():.4f}%")
print(f"  Min daily return: {(daily_returns * 100).values.flatten().min():.4f}%")
print(f"  Max daily return: {(daily_returns * 100).values.flatten().max():.4f}%")


In [ ]:
# Section 4: Compute CAGR for 1yr, 3yr, and 5yr Periods
"""
CAGR = (NAV_end / NAV_start)^(1/n) - 1
where n is the period in years.
"""

def compute_cagr(nav_series, periods_years):
    """
    Compute CAGR for specified periods in years.
    
    Parameters:
    -----------
    nav_series : pd.Series
        Sorted NAV values with datetime index
    periods_years : list
        List of periods in years (e.g., [1, 3, 5])
    
    Returns:
    --------
    dict : Dictionary with period as key and CAGR as value (as decimal)
    """
    cagr_dict = {}
    end_nav = nav_series.iloc[-1]
    
    for period in periods_years:
        # Calculate business days in period (approximately 252 per year)
        lookback_days = int(period * 252)
        
        if len(nav_series) > lookback_days:
            start_nav = nav_series.iloc[-lookback_days]
            cagr = (end_nav / start_nav) ** (1 / period) - 1
        else:
            # Not enough data for this period
            cagr = np.nan
        
        cagr_dict[f'{period}yr_cagr'] = cagr
    
    return cagr_dict

# Compute CAGR for all funds
cagr_results = []
periods = [1, 3, 5]

for amfi_code in nav_pivot.columns:
    fund_nav = nav_pivot[amfi_code].dropna()
    
    if len(fund_nav) > 0:
        fund_info = fund_master[fund_master['amfi_code'] == amfi_code].iloc[0]
        cagr_vals = compute_cagr(fund_nav, periods)
        
        row = {
            'amfi_code': amfi_code,
            'scheme_name': fund_info['scheme_name'],
            'fund_house': fund_info['fund_house'],
            'category': fund_info['category'],
        }
        row.update(cagr_vals)
        cagr_results.append(row)

cagr_report = pd.DataFrame(cagr_results)
cagr_report.to_csv(data_processed_dir / 'cagr_report.csv', index=False)

print(f"✓ CAGR report computed and saved: {data_processed_dir / 'cagr_report.csv'}")
print(f"\nCAGR Report (first 10 funds):")
print(cagr_report[['scheme_name', '1yr_cagr', '3yr_cagr', '5yr_cagr']].head(10).to_string())

print(f"\nCAGR Statistics:")
for period in periods:
    col_name = f'{period}yr_cagr'
    valid_cagr = cagr_report[col_name].dropna()
    if len(valid_cagr) > 0:
        print(f"\n{period}-Year CAGR:")
        print(f"  Mean: {valid_cagr.mean()*100:.2f}%")
        print(f"  Median: {valid_cagr.median()*100:.2f}%")
        print(f"  Std Dev: {valid_cagr.std()*100:.2f}%")
        print(f"  Range: {valid_cagr.min()*100:.2f}% to {valid_cagr.max()*100:.2f}%")


In [ ]:
# Section 5: Compute Sharpe Ratios
"""
Sharpe Ratio = (Rp - Rf) / Std(Rp) * sqrt(252)
Where:
  Rp = average daily return of fund
  Rf = risk-free rate = 6.5% annually = 6.5/252/100 daily
  Std(Rp) = standard deviation of daily returns
  252 = number of trading days in a year
"""

# Risk-free rate (6.5% annual = daily)
rf_annual = 0.065
rf_daily = rf_annual / 252

sharpe_results = []

for amfi_code in daily_returns.columns:
    fund_returns = daily_returns[amfi_code].dropna()
    
    if len(fund_returns) > 1:
        # Calculate annualized metrics
        avg_return = fund_returns.mean()
        std_return = fund_returns.std()
        
        # Sharpe ratio
        if std_return > 0:
            sharpe = ((avg_return - rf_daily) / std_return) * np.sqrt(252)
        else:
            sharpe = np.nan
        
        fund_info = fund_master[fund_master['amfi_code'] == amfi_code].iloc[0]
        
        row = {
            'amfi_code': amfi_code,
            'scheme_name': fund_info['scheme_name'],
            'fund_house': fund_info['fund_house'],
            'category': fund_info['category'],
            'sharpe_ratio': sharpe,
            'avg_daily_return_pct': avg_return * 100,
            'annual_return_pct': avg_return * 252 * 100,
            'daily_std_pct': std_return * 100,
            'annual_std_pct': std_return * np.sqrt(252) * 100,
        }
        sharpe_results.append(row)

sharpe_df = pd.DataFrame(sharpe_results)
sharpe_df.to_csv(data_processed_dir / 'sharpe_values.csv', index=False)

print(f"✓ Sharpe ratios computed and saved: {data_processed_dir / 'sharpe_values.csv'}")
print(f"\nSharpe Ratio Report (first 10 funds, sorted by Sharpe):")
print(sharpe_df.nlargest(10, 'sharpe_ratio')[['scheme_name', 'sharpe_ratio', 'annual_return_pct', 'annual_std_pct']].to_string())

print(f"\nSharpe Ratio Statistics:")
valid_sharpe = sharpe_df['sharpe_ratio'].dropna()
print(f"  Mean: {valid_sharpe.mean():.4f}")
print(f"  Median: {valid_sharpe.median():.4f}")
print(f"  Std Dev: {valid_sharpe.std():.4f}")
print(f"  Range: {valid_sharpe.min():.4f} to {valid_sharpe.max():.4f}")
print(f"  Best Sharpe: {valid_sharpe.max():.4f} ({sharpe_df.loc[valid_sharpe.idxmax(), 'scheme_name']})")
print(f"  Worst Sharpe: {valid_sharpe.min():.4f} ({sharpe_df.loc[valid_sharpe.idxmin(), 'scheme_name']})")


In [ ]:
# Section 6: Compute Sortino Ratios
"""
Sortino Ratio = (Rp - Rf) / Downside_Std * sqrt(252)
Where:
  Downside_Std = standard deviation of only negative returns (downside volatility)
  Only days where return < 0 are considered for downside volatility
"""

sortino_results = []

for amfi_code in daily_returns.columns:
    fund_returns = daily_returns[amfi_code].dropna()
    
    if len(fund_returns) > 1:
        # Calculate metrics
        avg_return = fund_returns.mean()
        
        # Downside standard deviation - only negative returns
        downside_returns = fund_returns[fund_returns < 0]
        
        if len(downside_returns) > 0:
            downside_std = downside_returns.std()
        else:
            downside_std = 0
        
        # Sortino ratio
        if downside_std > 0:
            sortino = ((avg_return - rf_daily) / downside_std) * np.sqrt(252)
        else:
            sortino = np.nan
        
        fund_info = fund_master[fund_master['amfi_code'] == amfi_code].iloc[0]
        
        row = {
            'amfi_code': amfi_code,
            'scheme_name': fund_info['scheme_name'],
            'fund_house': fund_info['fund_house'],
            'category': fund_info['category'],
            'sortino_ratio': sortino,
            'avg_daily_return_pct': avg_return * 100,
            'downside_std_daily_pct': downside_std * 100,
            'downside_std_annual_pct': downside_std * np.sqrt(252) * 100,
            'num_negative_days': len(downside_returns),
            'pct_negative_days': len(downside_returns) / len(fund_returns) * 100,
        }
        sortino_results.append(row)

sortino_df = pd.DataFrame(sortino_results)
sortino_df.to_csv(data_processed_dir / 'sortino_values.csv', index=False)

print(f"✓ Sortino ratios computed and saved: {data_processed_dir / 'sortino_values.csv'}")
print(f"\nSortino Ratio Report (first 10 funds, sorted by Sortino):")
print(sortino_df.nlargest(10, 'sortino_ratio')[['scheme_name', 'sortino_ratio', 'downside_std_annual_pct', 'pct_negative_days']].to_string())

print(f"\nSortino Ratio Statistics:")
valid_sortino = sortino_df['sortino_ratio'].dropna()
print(f"  Mean: {valid_sortino.mean():.4f}")
print(f"  Median: {valid_sortino.median():.4f}")
print(f"  Std Dev: {valid_sortino.std():.4f}")
print(f"  Range: {valid_sortino.min():.4f} to {valid_sortino.max():.4f}")
print(f"  Best Sortino: {valid_sortino.max():.4f} ({sortino_df.loc[valid_sortino.idxmax(), 'scheme_name']})")
print(f"  Worst Sortino: {valid_sortino.min():.4f} ({sortino_df.loc[valid_sortino.idxmin(), 'scheme_name']})")


In [ ]:
# Section 7: Compute Alpha and Beta vs Nifty 100
"""
Using scipy.stats.linregress to compute Beta and Alpha.
Beta = slope of regression (fund_returns vs benchmark_returns)
Alpha = intercept * 252 (annualized)
"""

# Get Nifty 100 benchmark data
nifty100 = benchmark_indices[benchmark_indices['index_name'] == 'NIFTY100'].copy()
nifty100 = nifty100.sort_values('date').reset_index(drop=True)

# Compute Nifty 100 returns
nifty100_pivot = nifty100.set_index('date')['close_value'].sort_index()
nifty100_returns = nifty100_pivot.pct_change().dropna()

print(f"Nifty 100 returns shape: {nifty100_returns.shape}")
print(f"Nifty 100 annualized return: {nifty100_returns.mean() * 252 * 100:.2f}%")
print(f"Nifty 100 annualized volatility: {nifty100_returns.std() * np.sqrt(252) * 100:.2f}%")

# Compute Alpha and Beta for each fund
alpha_beta_results = []

for amfi_code in daily_returns.columns:
    fund_returns = daily_returns[amfi_code].dropna()
    
    # Align returns - use common dates
    common_dates = fund_returns.index.intersection(nifty100_returns.index)
    
    if len(common_dates) > 10:  # Need at least 10 common observations
        fund_ret_aligned = fund_returns.loc[common_dates].values
        nifty_ret_aligned = nifty100_returns.loc[common_dates].values
        
        # Linear regression
        slope, intercept, r_value, p_value, std_err = stats.linregress(nifty_ret_aligned, fund_ret_aligned)
        
        # Annualize alpha
        alpha_annualized = float(intercept) * 252
        
        fund_info = fund_master[fund_master['amfi_code'] == amfi_code].iloc[0]
        
        row = {
            'amfi_code': amfi_code,
            'scheme_name': fund_info['scheme_name'],
            'fund_house': fund_info['fund_house'],
            'category': fund_info['category'],
            'beta': slope,
            'alpha_annualized_pct': alpha_annualized * 100,
            'intercept_daily': float(intercept),
            'r_squared': float(r_value) ** 2,
            'correlation': float(r_value),
            'num_observations': len(common_dates),
        }
        alpha_beta_results.append(row)

alpha_beta_df = pd.DataFrame(alpha_beta_results)
alpha_beta_df.to_csv(data_processed_dir / 'alpha_beta.csv', index=False)

print(f"\n✓ Alpha and Beta computed and saved: {data_processed_dir / 'alpha_beta.csv'}")
print(f"\nAlpha vs Nifty 100 (first 10 funds, sorted by Alpha):")
print(alpha_beta_df.nlargest(10, 'alpha_annualized_pct')[['scheme_name', 'alpha_annualized_pct', 'beta', 'r_squared']].to_string())

print(f"\nAlpha Statistics:")
valid_alpha = alpha_beta_df['alpha_annualized_pct'].dropna()
print(f"  Mean: {valid_alpha.mean():.2f}%")
print(f"  Median: {valid_alpha.median():.2f}%")
print(f"  Std Dev: {valid_alpha.std():.2f}%")
print(f"  Range: {valid_alpha.min():.2f}% to {valid_alpha.max():.2f}%")

print(f"\nBeta Statistics:")
valid_beta = alpha_beta_df['beta'].dropna()
print(f"  Mean: {valid_beta.mean():.4f}")
print(f"  Median: {valid_beta.median():.4f}")
print(f"  Std Dev: {valid_beta.std():.4f}")
print(f"  Range: {valid_beta.min():.4f} to {valid_beta.max():.4f}")


In [ ]:
# Section 8: Compute Maximum Drawdown and Worst Drawdown Dates
"""
Maximum Drawdown = min(NAV / running_max_NAV - 1)
Identifies the worst peak-to-trough decline for each fund.
"""

def compute_max_drawdown_with_dates(nav_series):
    """
    Compute maximum drawdown and identify drawdown period.
    
    Parameters:
    -----------
    nav_series : pd.Series
        NAV values with datetime index
    
    Returns:
    --------
    tuple : (max_drawdown, peak_date, trough_date, peak_nav, trough_nav)
    """
    running_max = nav_series.expanding().max()
    drawdown = (nav_series / running_max) - 1
    
    max_dd = drawdown.min()
    trough_idx = drawdown.idxmin()
    
    # Find the peak that led to this trough
    peak_idx = running_max[:trough_idx].idxmax()
    
    peak_nav = nav_series[peak_idx]
    trough_nav = nav_series[trough_idx]
    
    return max_dd, peak_idx, trough_idx, peak_nav, trough_nav

# Get NAV history with dates as index
nav_history_indexed = nav_history.set_index('date')

max_drawdown_results = []

for amfi_code in nav_pivot.columns:
    fund_nav = nav_pivot[amfi_code].dropna()
    
    if len(fund_nav) > 10:
        max_dd, peak_date, trough_date, peak_nav, trough_nav = compute_max_drawdown_with_dates(fund_nav)
        
        fund_info = fund_master[fund_master['amfi_code'] == amfi_code].iloc[0]
        
        row = {
            'amfi_code': amfi_code,
            'scheme_name': fund_info['scheme_name'],
            'fund_house': fund_info['fund_house'],
            'category': fund_info['category'],
            'max_drawdown_pct': max_dd * 100,
            'peak_date': peak_date.strftime('%Y-%m-%d'),
            'trough_date': trough_date.strftime('%Y-%m-%d'),
            'peak_nav': peak_nav,
            'trough_nav': trough_nav,
            'drawdown_duration_days': (trough_date - peak_date).days,
        }
        max_drawdown_results.append(row)

max_drawdown_df = pd.DataFrame(max_drawdown_results)
max_drawdown_df.to_csv(data_processed_dir / 'max_drawdown.csv', index=False)

print(f"✓ Maximum drawdown computed and saved: {data_processed_dir / 'max_drawdown.csv'}")
print(f"\nWorst Drawdown (first 10 funds, sorted by drawdown severity):")
print(max_drawdown_df.nsmallest(10, 'max_drawdown_pct')[['scheme_name', 'max_drawdown_pct', 'peak_date', 'trough_date', 'drawdown_duration_days']].to_string())

print(f"\nMaximum Drawdown Statistics:")
valid_dd = max_drawdown_df['max_drawdown_pct'].dropna()
print(f"  Mean: {valid_dd.mean():.2f}%")
print(f"  Median: {valid_dd.median():.2f}%")
print(f"  Std Dev: {valid_dd.std():.2f}%")
print(f"  Range: {valid_dd.min():.2f}% to {valid_dd.max():.2f}%")
print(f"  Worst Drawdown: {valid_dd.min():.2f}% ({max_drawdown_df.loc[valid_dd.idxmin(), 'scheme_name']})")
print(f"  Best (Least) Drawdown: {valid_dd.max():.2f}% ({max_drawdown_df.loc[valid_dd.idxmax(), 'scheme_name']})")


In [ ]:
# Section 9: Build Fund Scorecard (Composite Score 0-100)
"""
Fund Scorecard combines multiple metrics:
- 30% x 3yr CAGR rank (higher is better)
- 25% x Sharpe ratio rank (higher is better)
- 20% x Alpha rank (higher is better)
- 15% x expense_ratio rank (INVERSE - lower is better)
- 10% x max_drawdown rank (INVERSE - less negative is better)
"""

# Merge all metrics
scorecard_data = fund_master[['amfi_code', 'scheme_name', 'fund_house', 'category', 'expense_ratio_pct']].copy()

# Merge CAGR (3yr)
cagr_3yr = cagr_report[['amfi_code', '3yr_cagr']].copy()
cagr_3yr.columns = ['amfi_code', 'cagr_3yr']
scorecard_data = scorecard_data.merge(cagr_3yr, on='amfi_code', how='left')

# Merge Sharpe
sharpe_merge = sharpe_df[['amfi_code', 'sharpe_ratio']].copy()
scorecard_data = scorecard_data.merge(sharpe_merge, on='amfi_code', how='left')

# Merge Alpha
alpha_merge = alpha_beta_df[['amfi_code', 'alpha_annualized_pct']].copy()
alpha_merge.columns = ['amfi_code', 'alpha_pct']
scorecard_data = scorecard_data.merge(alpha_merge, on='amfi_code', how='left')

# Merge Max Drawdown
dd_merge = max_drawdown_df[['amfi_code', 'max_drawdown_pct']].copy()
scorecard_data = scorecard_data.merge(dd_merge, on='amfi_code', how='left')

print(f"Scorecard data shape: {scorecard_data.shape}")
print(f"Columns: {scorecard_data.columns.tolist()}")
print(f"Missing values:\n{scorecard_data.isnull().sum()}")

# Rank each metric
# For metrics where higher is better: rank 1 = best, 40 = worst
# For metrics where lower is better (expense ratio, drawdown): invert rank

# 3yr CAGR rank (higher better)
scorecard_data['cagr_rank'] = scorecard_data['cagr_3yr'].rank(method='min', na_option='bottom')

# Sharpe rank (higher better)
scorecard_data['sharpe_rank'] = scorecard_data['sharpe_ratio'].rank(method='min', na_option='bottom')

# Alpha rank (higher better)
scorecard_data['alpha_rank'] = scorecard_data['alpha_pct'].rank(method='min', na_option='bottom')

# Expense ratio rank (lower better, so invert)
scorecard_data['expense_rank_raw'] = scorecard_data['expense_ratio_pct'].rank(method='min', na_option='bottom')
scorecard_data['expense_rank'] = scorecard_data.shape[0] + 1 - scorecard_data['expense_rank_raw']

# Max drawdown rank (less negative better, so invert - higher drawdown should get lower score)
scorecard_data['dd_rank_raw'] = scorecard_data['max_drawdown_pct'].rank(method='min', na_option='bottom')
scorecard_data['dd_rank'] = scorecard_data.shape[0] + 1 - scorecard_data['dd_rank_raw']

# Normalize ranks to 0-100 scale
n_funds = scorecard_data.shape[0]

scorecard_data['cagr_score'] = (scorecard_data['cagr_rank'] / n_funds) * 100
scorecard_data['sharpe_score'] = (scorecard_data['sharpe_rank'] / n_funds) * 100
scorecard_data['alpha_score'] = (scorecard_data['alpha_rank'] / n_funds) * 100
scorecard_data['expense_score'] = (scorecard_data['expense_rank'] / n_funds) * 100
scorecard_data['dd_score'] = (scorecard_data['dd_rank'] / n_funds) * 100

# Calculate composite score
weights = {
    'cagr_score': 0.30,
    'sharpe_score': 0.25,
    'alpha_score': 0.20,
    'expense_score': 0.15,
    'dd_score': 0.10,
}

scorecard_data['composite_score'] = (
    scorecard_data['cagr_score'] * weights['cagr_score'] +
    scorecard_data['sharpe_score'] * weights['sharpe_score'] +
    scorecard_data['alpha_score'] * weights['alpha_score'] +
    scorecard_data['expense_score'] * weights['expense_score'] +
    scorecard_data['dd_score'] * weights['dd_score']
)

# Rank by composite score
scorecard_data['overall_rank'] = scorecard_data['composite_score'].rank(method='min', ascending=False)

# Select columns to export
scorecard_export = scorecard_data[[
    'amfi_code', 'scheme_name', 'fund_house', 'category', 'expense_ratio_pct',
    'cagr_3yr', 'sharpe_ratio', 'alpha_pct', 'max_drawdown_pct',
    'cagr_score', 'sharpe_score', 'alpha_score', 'expense_score', 'dd_score',
    'composite_score', 'overall_rank'
]].copy()

scorecard_export.to_csv(data_processed_dir / 'fund_scorecard.csv', index=False)

print(f"\n✓ Fund Scorecard created and saved: {data_processed_dir / 'fund_scorecard.csv'}")
print(f"\nTop 10 Funds by Composite Score:")
print(scorecard_export.head(10)[['overall_rank', 'scheme_name', 'composite_score', 'cagr_3yr', 'sharpe_ratio', 'alpha_pct', 'expense_ratio_pct', 'max_drawdown_pct']].to_string())

print(f"\nBottom 10 Funds by Composite Score:")
print(scorecard_export.tail(10)[['overall_rank', 'scheme_name', 'composite_score', 'cagr_3yr', 'sharpe_ratio', 'alpha_pct', 'expense_ratio_pct', 'max_drawdown_pct']].to_string())

print(f"\nComposite Score Statistics:")
print(f"  Mean: {scorecard_export['composite_score'].mean():.2f}")
print(f"  Median: {scorecard_export['composite_score'].median():.2f}")
print(f"  Std Dev: {scorecard_export['composite_score'].std():.2f}")
print(f"  Range: {scorecard_export['composite_score'].min():.2f} to {scorecard_export['composite_score'].max():.2f}")


In [ ]:
# Section 10: Plot Benchmark Comparison for Top Funds
"""
Plot top 5 funds (by scorecard) vs NIFTY50 and NIFTY100 over 3 years (2023-2026).
Index all to 100 at start. Compute tracking error.
"""

# Get top 5 funds
top_5_funds = scorecard_export.head(5)['amfi_code'].tolist()

print(f"Top 5 funds for benchmark comparison:")
for idx, fund_code in enumerate(top_5_funds, 1):
    fund_name = scorecard_export[scorecard_export['amfi_code'] == fund_code]['scheme_name'].values[0]
    score = scorecard_export[scorecard_export['amfi_code'] == fund_code]['composite_score'].values[0]
    print(f"  {idx}. {fund_name} (Score: {score:.2f})")

# Filter data to 3-year window (2023-01-01 to 2026-05-29)
start_date = pd.Timestamp('2023-01-01')
end_date = nav_history['date'].max()

# Filter NAV data
nav_filtered = nav_history[(nav_history['date'] >= start_date) & (nav_history['date'] <= end_date)].copy()

# Filter benchmark data
bench_filtered = benchmark_indices[(benchmark_indices['date'] >= start_date) & (benchmark_indices['date'] <= end_date)].copy()

print(f"\nFiltered data range: {nav_filtered['date'].min()} to {nav_filtered['date'].max()}")

# Pivot NAV data
nav_pivot_filtered = nav_filtered.pivot_table(index='date', columns='amfi_code', values='nav')

# Get top 5 fund NAVs
top_5_navs = nav_pivot_filtered[top_5_funds].copy()

# Index to 100 at start
top_5_indexed = (top_5_navs / top_5_navs.iloc[0]) * 100

# Get NIFTY50 and NIFTY100
nifty50 = bench_filtered[bench_filtered['index_name'] == 'NIFTY50'].copy()
nifty100 = bench_filtered[bench_filtered['index_name'] == 'NIFTY100'].copy()

# Pivot benchmarks
nifty50_pivot = nifty50.set_index('date')['close_value'].sort_index()
nifty100_pivot = nifty100.set_index('date')['close_value'].sort_index()

# Index benchmarks to 100 at start
nifty50_indexed = (nifty50_pivot / nifty50_pivot.iloc[0]) * 100
nifty100_indexed = (nifty100_pivot / nifty100_pivot.iloc[0]) * 100

# Create plot
fig, ax = plt.subplots(figsize=(14, 8))

# Plot top 5 funds
colors_funds = plt.get_cmap('Set1')(np.linspace(0, 1, 5))
for i, fund_code in enumerate(top_5_funds):
    fund_name = scorecard_export[scorecard_export['amfi_code'] == fund_code]['scheme_name'].values[0]
    ax.plot(top_5_indexed.index, top_5_indexed[fund_code], label=f'Top {i+1}: {fund_name[:40]}...', 
            linewidth=2.5, color=colors_funds[i])

# Plot benchmarks with distinct styles
ax.plot(nifty50_indexed.index, nifty50_indexed, label='NIFTY 50', 
        linewidth=2.5, color='red', linestyle='--', alpha=0.8)
ax.plot(nifty100_indexed.index, nifty100_indexed, label='NIFTY 100', 
        linewidth=2.5, color='orange', linestyle='--', alpha=0.8)

ax.set_xlabel('Date', fontsize=12, fontweight='bold')
ax.set_ylabel('Indexed Value (100 at start)', fontsize=12, fontweight='bold')
ax.set_title('Top 5 Funds vs NIFTY Benchmarks (Indexed, 2023-2026)', fontsize=14, fontweight='bold')
ax.legend(loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_facecolor('#f8f9fa')

plt.tight_layout()
plt.savefig(charts_dir / 'benchmark_comparison.png', dpi=300, bbox_inches='tight')
plt.close()

print(f"\n✓ Benchmark comparison chart saved: {charts_dir / 'benchmark_comparison.png'}")

# Compute tracking error vs benchmarks
print(f"\n\nTracking Error Analysis:")
print("=" * 80)

for fund_code in top_5_funds:
    fund_name = scorecard_export[scorecard_export['amfi_code'] == fund_code]['scheme_name'].values[0]
    
    # Get fund returns
    fund_nav = nav_pivot_filtered[fund_code].dropna()
    fund_returns = fund_nav.pct_change().dropna()
    
    # Align with benchmarks
    common_dates = fund_returns.index.intersection(nifty50_indexed.index)
    
    if len(common_dates) > 10:
        fund_ret = fund_returns.loc[common_dates].values
        nifty50_ret = nifty50_indexed.loc[common_dates].pct_change().dropna().values
        nifty100_ret = nifty100_indexed.loc[common_dates].pct_change().dropna().values
        
        # Align lengths
        min_len = min(len(fund_ret), len(nifty50_ret), len(nifty100_ret))
        fund_ret = fund_ret[-min_len:]
        nifty50_ret = nifty50_ret[-min_len:]
        nifty100_ret = nifty100_ret[-min_len:]
        
        # Compute tracking error
        te_nifty50 = np.std(fund_ret - nifty50_ret) * np.sqrt(252)
        te_nifty100 = np.std(fund_ret - nifty100_ret) * np.sqrt(252)
        
        # Compute excess returns
        excess_nifty50 = (fund_ret.mean() - nifty50_ret.mean()) * 252 * 100
        excess_nifty100 = (fund_ret.mean() - nifty100_ret.mean()) * 252 * 100
        
        print(f"\n{fund_name[:50]}:")
        print(f"  Tracking Error vs NIFTY50: {te_nifty50*100:.2f}%")
        print(f"  Tracking Error vs NIFTY100: {te_nifty100*100:.2f}%")
        print(f"  Excess Return vs NIFTY50: {excess_nifty50:.2f}%")
        print(f"  Excess Return vs NIFTY100: {excess_nifty100:.2f}%")


In [ ]:
# Section 11: Verify All Output Files and PASS/FAIL Checklist

# Import required libraries
import pandas as pd
import numpy as np
from pathlib import Path
import os

# Redefine paths in case they're not available
current_dir = Path.cwd()
if (current_dir / 'data' / 'processed').exists():
    project_root = current_dir
elif current_dir.name == 'notebooks' and (current_dir.parent / 'data' / 'processed').exists():
    project_root = current_dir.parent
elif (current_dir.parent / 'data' / 'processed').exists():
    project_root = current_dir.parent
else:
    project_root = current_dir

data_processed_dir = project_root / 'data' / 'processed'
charts_dir = project_root / 'reports' / 'charts'

print(f"Current working directory: {os.getcwd()}")
print(f"Project root: {project_root}")
print(f"Data processed dir: {data_processed_dir}")
print(f"Data processed dir exists: {data_processed_dir.exists()}")

print("\n" + "="*80)
print("DAY 4: FUND PERFORMANCE ANALYTICS - COMPLETION CHECKLIST")
print("="*80)

# Expected output files
expected_files = {
    'returns_computed.csv': 'Daily returns for all 40 funds',
    'cagr_report.csv': 'CAGR for 1yr, 3yr, 5yr periods',
    'sharpe_values.csv': 'Sharpe ratios',
    'sortino_values.csv': 'Sortino ratios',
    'alpha_beta.csv': 'Alpha and Beta vs Nifty 100',
    'max_drawdown.csv': 'Maximum Drawdown analysis',
    'fund_scorecard.csv': 'Composite Fund Scorecard (0-100)',
}

print("\n[CSV Output Files]")
for filename, description in expected_files.items():
    filepath = data_processed_dir / filename
    if filepath.exists():
        file_size = filepath.stat().st_size
        num_rows = len(pd.read_csv(filepath))
        print(f"✓ PASS: {filename}")
        print(f"         {description}")
        print(f"         Size: {file_size:,} bytes | Rows: {num_rows}")
    else:
        print(f"✗ FAIL: {filename} - FILE NOT FOUND")

# Check chart
print("\n[Visualization Output]")
chart_file = charts_dir / 'benchmark_comparison.png'
if chart_file.exists():
    file_size = chart_file.stat().st_size
    print(f"✓ PASS: benchmark_comparison.png")
    print(f"         Size: {file_size:,} bytes")
else:
    print(f"✗ FAIL: benchmark_comparison.png - FILE NOT FOUND")

# Verify data integrity
print("\n[Data Integrity Checks]")

try:
    # Check 1: Daily returns distribution
    returns_df = pd.read_csv(data_processed_dir / 'returns_computed.csv')
    print(f"✓ Daily returns shape: {returns_df.shape}")
except Exception as e:
    print(f"✗ Error loading returns: {e}")
    returns_df = None

try:
    # Check 2: CAGR report has 40 funds
    cagr_df = pd.read_csv(data_processed_dir / 'cagr_report.csv')
    print(f"✓ CAGR report has {len(cagr_df)} funds (expected 40)")
except Exception as e:
    print(f"✗ Error loading CAGR: {e}")
    cagr_df = None

try:
    # Check 3: Sharpe ratio range (-5 to +5 is typical)
    sharpe_df_check = pd.read_csv(data_processed_dir / 'sharpe_values.csv')
    sharpe_vals = sharpe_df_check['sharpe_ratio'].dropna()
    print(f"✓ Sharpe ratio range: {sharpe_vals.min():.4f} to {sharpe_vals.max():.4f}")
except Exception as e:
    print(f"✗ Error loading Sharpe: {e}")
    sharpe_df_check = None

try:
    # Check 4: Alpha reasonable range
    alpha_df_check = pd.read_csv(data_processed_dir / 'alpha_beta.csv')
    alpha_vals = alpha_df_check['alpha_annualized_pct'].dropna()
    print(f"✓ Alpha range: {alpha_vals.min():.2f}% to {alpha_vals.max():.2f}%")
except Exception as e:
    print(f"✗ Error loading Alpha/Beta: {e}")
    alpha_df_check = None

if alpha_df_check is not None:
    try:
        # Check 5: Beta typically 0 to 2
        beta_vals = alpha_df_check['beta'].dropna()
        print(f"✓ Beta range: {beta_vals.min():.4f} to {beta_vals.max():.4f}")
    except Exception as e:
        print(f"✗ Error with Beta: {e}")

try:
    # Check 6: Max drawdown in reasonable range
    dd_df_check = pd.read_csv(data_processed_dir / 'max_drawdown.csv')
    dd_vals = dd_df_check['max_drawdown_pct'].dropna()
    print(f"✓ Max drawdown range: {dd_vals.min():.2f}% to {dd_vals.max():.2f}%")
except Exception as e:
    print(f"✗ Error loading Max Drawdown: {e}")
    dd_df_check = None

try:
    # Check 7: Composite score 0-100
    scorecard_df_check = pd.read_csv(data_processed_dir / 'fund_scorecard.csv')
    score_vals = scorecard_df_check['composite_score'].dropna()
    print(f"✓ Composite score range: {score_vals.min():.2f} to {score_vals.max():.2f}")
    print(f"✓ Fund scorecard has {len(scorecard_df_check)} funds (expected 40)")
except Exception as e:
    print(f"✗ Error loading Fund Scorecard: {e}")
    scorecard_df_check = None

print("\n" + "="*80)
print("REQUIREMENTS SUMMARY")
print("="*80)

requirements = [
    ("Task 1", "Compute daily returns", returns_df is not None and len(returns_df) > 0),
    ("Task 2", "Compute CAGR (1yr, 3yr, 5yr)", cagr_df is not None and len(cagr_df) == 40),
    ("Task 3", "Compute Sharpe Ratio", sharpe_df_check is not None and len(sharpe_df_check) == 40),
    ("Task 4", "Compute Sortino Ratio", (data_processed_dir / 'sortino_values.csv').exists()),
    ("Task 5", "Compute Alpha and Beta", alpha_df_check is not None and len(alpha_df_check) > 0),
    ("Task 6", "Compute Maximum Drawdown", dd_df_check is not None and len(dd_df_check) == 40),
    ("Task 7", "Build Fund Scorecard", scorecard_df_check is not None and len(scorecard_df_check) == 40),
    ("Task 8", "Benchmark comparison chart", chart_file.exists()),
]

pass_count = 0
for task_id, task_name, status in requirements:
    status_str = "✓ PASS" if status else "✗ FAIL"
    print(f"{status_str}: {task_id} - {task_name}")
    if status:
        pass_count += 1

print("\n" + "="*80)
print(f"OVERALL RESULT: {pass_count}/{len(requirements)} TASKS PASSED")
print("="*80)

if pass_count == len(requirements):
    print("\n🎉 ALL DAY 4 REQUIREMENTS COMPLETED SUCCESSFULLY!")
else:
    print(f"\n⚠️  {len(requirements) - pass_count} task(s) need attention.")
